In [35]:
import random
import copy
from itertools import combinations_with_replacement

TODAS = list(combinations_with_replacement(range(7), 2))   # 28 fichas del doble 6

print(f"Total de fichas: {len(TODAS)}")
print(TODAS)

Total de fichas: 28
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (3, 3), (3, 4), (3, 5), (3, 6), (4, 4), (4, 5), (4, 6), (5, 5), (5, 6), (6, 6)]


In [36]:
# ---------- Motor de simulación ----------

def _repartir(desconocidas, cantidades, sin_numeros, intentos=200):
    """Reparte al azar las fichas desconocidas respetando cantidades y pases."""
    jugadores = list(cantidades)
    for _ in range(intentos):
        random.shuffle(jugadores)           # evita favorecer siempre al mismo jugador
        pool = desconocidas[:]
        random.shuffle(pool)
        reparto = {}
        for j in jugadores:
            n, prohibidos = cantidades[j], sin_numeros[j]
            mano = [f for f in pool if f[0] not in prohibidos and f[1] not in prohibidos][:n]
            if len(mano) < n:
                break
            reparto[j] = mano
            pool = [f for f in pool if f not in mano]
        else:
            return reparto
    return None


def _simular(manos, extremos, turno, mi_equipo, pases):
    """Termina la partida con todos jugando al azar.
    Devuelve 1 (gana mi equipo), 0 (pierde) o 0.5 (empate)."""
    manos = [list(m) for m in manos]
    izq, der = extremos
    while True:
        mano = manos[turno]
        opciones = []
        for f in mano:
            a, b = f
            if a == izq or b == izq:
                opciones.append((f, 0))
            if izq != der and (a == der or b == der):
                opciones.append((f, 1))
        if opciones:
            f, lado = random.choice(opciones)
            mano.remove(f)
            if lado == 0:
                izq = f[1] if f[0] == izq else f[0]
            else:
                der = f[1] if f[0] == der else f[0]
            pases = 0
            if not mano:
                return 1.0 if turno % 2 == mi_equipo else 0.0
        else:
            pases += 1
            if pases == 4:                  # trancado: gana quien tenga menos puntos
                pts = [sum(a + b for a, b in m) for m in manos]
                menor = min(pts)
                eq = {i % 2 for i, p in enumerate(pts) if p == menor}
                if len(eq) == 1:
                    return 1.0 if eq.pop() == mi_equipo else 0.0
                return 0.5
        turno = (turno + 1) % 4


def _aplicar(extremos, ficha, lado):
    izq, der = extremos
    if lado == 'i':
        return ((ficha[1] if ficha[0] == izq else ficha[0]), der)
    return (izq, (ficha[1] if ficha[0] == der else ficha[0]))



In [37]:
# ---------- Partida (entrada de jugadas) ----------

class Partida:
    def __init__(self, mi_posicion, mi_mano, empieza, simulaciones=3000):
        self.pos = mi_posicion
        self.mano = [tuple(sorted(f)) for f in mi_mano]
        self.turno = empieza
        self.sims = simulaciones
        self.extremos = None
        self.jugadas = []
        self.sin = {i: set() for i in range(4)}      # números que cada jugador NO tiene
        self.cuenta = {i: 0 for i in range(4)}       # fichas jugadas por jugador
        self.pases = 0                               # pases seguidos
        self.snaps = []
        quien = "Tienes el 6-6." if empieza == mi_posicion else f"Abre el jugador {empieza}."
        print(f"{quien} Escribe 66 para empezar.")

    def _estado(self):
        return (self.extremos, self.turno, self.mano, self.jugadas,
                self.sin, self.cuenta, self.pases)

    def deshacer(self):
        if not self.snaps:
            print("Nada que deshacer.")
            return
        (self.extremos, self.turno, self.mano, self.jugadas,
         self.sin, self.cuenta, self.pases) = self.snaps.pop()
        print(f"Última jugada deshecha. Turno del jugador {self.turno}.")

    def _una(self, tok):
        tok = tok.strip().lower()
        jugador = self.turno

        if tok in ("p", "pasa"):
            if self.extremos is None:
                print("No se puede pasar al inicio.")
                return False
            if jugador == self.pos and any(self.extremos[0] in f or self.extremos[1] in f for f in self.mano):
                print("Tienes fichas que entran: no puedes pasar.")
                return False
            self.snaps.append(copy.deepcopy(self._estado()))
            self.sin[jugador].update(self.extremos)
            self.pases += 1
            self.turno = (jugador + 1) % 4
            return True

        if len(tok) < 2 or not tok[:2].isdigit() or max(int(tok[0]), int(tok[1])) > 6:
            print(f"No entendí '{tok}'. Escribe la ficha pegada (ej: 36) o p para pasar.")
            return False

        ficha = tuple(sorted((int(tok[0]), int(tok[1]))))
        lado = tok[2] if len(tok) > 2 else None

        if ficha in self.jugadas:
            print(f"El {ficha} ya se jugó.")
            return False
        if jugador == self.pos and ficha not in self.mano:
            print(f"Es tu turno y el {ficha} no está en tu mano.")
            return False
        if jugador != self.pos and ficha in self.mano:
            print(f"El {ficha} es tuyo, no lo pudo jugar el jugador {jugador}.")
            return False
        if set(ficha) & self.sin[jugador]:
            print(f"El jugador {jugador} ya había pasado en {sorted(self.sin[jugador])}: no puede tener el {ficha}.")
            return False

        if self.extremos is None:
            if ficha != (6, 6):
                print("La partida abre con el 6-6.")
                return False
            nuevo = (6, 6)
        else:
            izq, der = self.extremos
            en_i, en_d = izq in ficha, der in ficha
            if not (en_i or en_d):
                print(f"El {ficha} no entra en la mesa {self.extremos}.")
                return False
            if izq == der or (en_i and not en_d):
                lado = 'i'
            elif en_d and not en_i:
                lado = 'd'
            elif lado not in ('i', 'd'):
                print(f"El {ficha} entra por los dos lados {self.extremos}. Escribe i o d (ej: {tok[:2]}i).")
                return False
            nuevo = _aplicar(self.extremos, ficha, lado)

        self.snaps.append(copy.deepcopy(self._estado()))
        self.extremos = nuevo
        self.jugadas = self.jugadas + [ficha]
        self.cuenta[jugador] += 1
        self.pases = 0
        if jugador == self.pos:
            self.mano = [f for f in self.mano if f != ficha]
        self.turno = (jugador + 1) % 4
        return True

    def j(self, texto):
        """Una o varias jugadas: p.j('36') o p.j('66 p 45d 55')"""
        for tok in texto.replace(",", " ").split():
            if not self._una(tok):
                return
        if self.extremos is not None:
            print(f"\nMesa: {self.extremos} | Tu mano: {self.mano}")
        if not self.mano:
            print("¡Te quedaste sin fichas! Ganó tu equipo.")
            return
        if self.turno == self.pos and self.extremos is not None:
            self.recomendar()
        else:
            print(f"Turno del jugador {self.turno}.")


    # ---------- Recomendación ----------

    def recomendar(self):
        mi_equipo = self.pos % 2
        conocidas = set(self.mano) | set(self.jugadas)
        desconocidas = [f for f in TODAS if f not in conocidas]
        cantidades = {j: 7 - self.cuenta[j] for j in range(4) if j != self.pos}

        if sum(cantidades.values()) != len(desconocidas):
            print("Error: las fichas no cuadran. Revisa tu mano o el historial (usa 'deshacer').")
            return

        izq, der = self.extremos
        opciones = []
        for f in self.mano:
            if izq in f:
                opciones.append((f, 'i'))
            if der in f and izq != der:
                opciones.append((f, 'd'))
        if not opciones:
            print("No tienes jugadas válidas: escribe p para pasar.")
            return

        # Mismos repartos para todas las jugadas: comparación más justa y más rápida
        repartos = [r for r in (_repartir(desconocidas, cantidades, self.sin)
                                for _ in range(self.sims)) if r]
        if not repartos:
            print("No pude generar repartos compatibles. Revisa los pases registrados.")
            return

        resultados = []
        sig = (self.pos + 1) % 4
        for f, lado in opciones:
            nuevo = _aplicar(self.extremos, f, lado)
            resto = [x for x in self.mano if x != f]
            total = 0.0
            for r in repartos:
                if not resto:
                    total += 1.0
                    continue
                manos = [None] * 4
                manos[self.pos] = resto
                for j, m in r.items():
                    manos[j] = m
                total += _simular(manos, nuevo, sig, mi_equipo, 0)
            p = total / len(repartos)
            err = 1.96 * (p * (1 - p) / len(repartos)) ** 0.5
            codigo = f"{f[0]}{f[1]}" + (lado if (izq != der and izq in f and der in f) else "")
            resultados.append((p, err, codigo, f, nuevo))

        print(f"\nProbabilidad de ganar ({len(repartos)} simulaciones por jugada):\n")
        for p, err, codigo, f, nuevo in sorted(resultados, reverse=True):
            print(f"  {codigo:<4} {f}  -> mesa {nuevo}:  {p:.1%} (±{err:.1%})")

    def jugar(self):
        """Modo interactivo. Escribe cada jugada y Enter. 'deshacer' corrige, 'fin' sale."""
        while True:
            marca = " (TÚ)" if self.turno == self.pos else ""
            t = input(f"Jugador {self.turno}{marca} >> ").strip().lower()
            if t in ("fin", "salir"):
                break
            if t == "deshacer":
                self.deshacer()
            elif t:
                self.j(t)




In [38]:
# ---------- Uso ----------
p = Partida(mi_posicion=2,
             mi_mano=[(0,6), (3,6), (2,6), (1,2), (0,1), (4,5), (2,5)],
            empieza=1)
p.jugar()

Abre el jugador 1. Escribe 66 para empezar.


KeyboardInterrupt: Interrupted by user